In [ ]:
%run ./2_models.ipynb
%run ./3_tools.ipynb
%run ./4_utils.ipynb

# 스스로 판단하고 작업하는 멀티 에이전트 만들기

Chapter 14에서는 여러 에이전트가 협력하여 책의 목차를 작성하고, 작업 중 사용자에게 진행 상황을 보고하며 다음 요청을 받았습니다.  
이번 장에서는 사용자 요구사항을 분석하고 작성된 목차를 검토하는 에이전트를 추가합니다. 에이전트들이 작업 결과와 검토 의견을 공유하면서 후속 작업을 이어가고,  
사용자 의견이 필요하거나 반복 호출이 일정 횟수를 넘으면 사용자에게 돌아오도록 구성합니다.

## Chapter 14에서 달라진 점

| 변경 사항 | 설명 |
|---|---|
| 사용자 요구사항 분석 | `business_analyst`가 사용자 발언과 현재 작업 상황을 분석하여 `user_request`에 정리합니다. |
| 목차 검토 | `outline_reviewer`가 작성된 목차의 요구사항 충족 여부, 논리적 흐름, 참고 자료 등을 검토합니다. |
| 검토 의견 공유 | 목차 검토와 벡터 검색의 후속 작업 제안을 `ai_recommendation`에 저장해 비즈니스 분석가에게 전달합니다. |
| 목차 작성 프롬프트 개선 | `content_strategist`가 목차 작성 템플릿과 `user_request`를 바탕으로 목차를 작성하고 수정합니다. |
| 작업 흐름 변경 | 목차 작성 후 검토를 거치고, 검토나 벡터 검색이 끝나면 비즈니스 분석가가 현재 상황을 다시 분석합니다. |
| 반복 호출 제한 | Supervisor 호출 횟수를 기록하고, 기준을 초과하면 `communicator`를 통해 사용자에게 진행 상황을 보고합니다. |

## 추가된 공유 상태

| 필드 | 타입 | 설명 |
|---|---|---|
| `user_request` | `str` | 비즈니스 분석가가 분석하여 정리한 현재 요구사항 |
| `ai_recommendation` | `str` | 목차 검토 또는 벡터 검색 후 전달할 제안 |
| `supervisor_call_count` | `int` | Supervisor의 반복 호출 횟수 |

## 주요 함수

| 함수 | 핵심 역할 | 주요 결과 | 구분 |
|---|---|---|---|
| `business_analyst()` | 사용자 발언과 AI의 제안을 바탕으로 현재 요구사항 분석 | `user_request` 갱신 | 신규 |
| `outline_reviewer()` | 작성된 목차를 검토하고 개선 방향 제안 | `ai_recommendation` 갱신 | 신규 |
| `supervisor()` | 다음 작업과 담당 에이전트를 결정하고 반복 호출 횟수 확인 | 새로운 `Task` 생성 | 변경 |
| `supervisor_router()` | Supervisor의 결정에 따라 다음 실행 노드 선택 | 에이전트 이름 반환 | 기존 |
| `vector_search_agent()` | 벡터 DB에서 참고 문서를 검색하고 후속 작업 제안 | `references`, `ai_recommendation` 갱신 | 변경 |
| `content_strategist()` | 목차 작성 템플릿과 사용자 요구사항을 반영해 목차 작성 및 수정 | `outline.md` 저장 | 변경 |
| `web_search_agent()` | 웹 검색으로 외부 자료를 수집하고 벡터 DB에 추가 | Chroma DB 갱신 | 기존 |
| `communicator()` | 사용자에게 진행 상황을 보고하고 의견을 확인 | `messages` 갱신 및 호출 횟수 초기화 | 변경 |

## 주요 작업 흐름

```text
사용자 입력
    ↓
business_analyst → supervisor
                       ├→ content_strategist → outline_reviewer ─┐
                       ├→ web_search_agent → vector_search_agent ┤
                       ├→ vector_search_agent ───────────────────┤
                       └→ communicator → 종료                    │
                                                                 ↓
                                                          business_analyst
```

`business_analyst`는 목차 검토 의견이나 검색 결과를 분석한 뒤 Supervisor에 전달합니다.  
Supervisor는 다음 작업을 선택하며, 사용자와의 소통이 필요하거나 호출 횟수가 기준을 초과하면 `communicator`를 선택합니다.

In [2]:
import os 

current_path = os.getcwd()
filename = "multi_agent_tutorial.ipynb"
absolute_path = os.path.join(current_path, filename)

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-6-luna") # 모델 초기화

In [4]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict
from typing import List
#from models import Task
# %run ./models.ipynb로 불러온 Task 사용

# 상태 정의
class State(TypedDict):
    messages: List[AnyMessage | str]
    task_history: List[Task]    
    references: dict
    user_request: str           # chap15 사용자의 요구사항을 저장하는 변수
    ai_recommendation: str      # chap15 AI의 추천을 저장하는 변수
    supervisor_call_count: int  # chap15 supervisor 호출 횟수를 저장하는 변수

# chap 15 - 신규 에이전트 추가

## business_analyst() — 사용자 요구사항 및 목차 검토 결과 분석

사용자의 최근 입력과 AI 팀의 작업 상황을 바탕으로 현재 요구사항과 필요한 작업을 분석하는 에이전트입니다.  
목차 검토 에이전트가 작성한 `ai_recommendation`도 함께 살펴봅니다. 분석 결과는 `user_request`에 저장하여 Supervisor와 다른 에이전트가 이후 작업에 활용할 수 있도록 전달합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 대화 내용, 사용자 요구사항, 참고 자료 및 AI의 검토 의견을 담은 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자와 에이전트의 대화 및 작업 기록 |
| `user_request` | 비즈니스 분석가가 분석하여 정리한 현재 요구사항 |
| `references` | 요구사항과 목차를 분석할 때 참고할 자료 |
| `ai_recommendation` | `outline_reviewer`가 작성한 목차 검토 결과와 개선 제안 |

**반환값**

- `dict`: 갱신된 `messages`와 `user_request`를 반환합니다.
- 처리한 `ai_recommendation`은 빈 문자열로 초기화합니다.

**주요 처리 과정**

1. 사용자 요구사항과 AI의 검토 의견을 분석할 시스템 프롬프트를 정의합니다.
2. `PromptTemplate`, LLM, `StrOutputParser`를 연결하여 분석 체인을 구성합니다.
3. `messages`에서 사용자의 가장 최근 발언을 찾습니다.
4. `ai_recommendation`, 참고 자료, 현재 목차와 대화 내용을 입력 데이터로 준비합니다.
5. 이전 요청이 달성되었는지와 현재 필요한 작업이 무엇인지 분석합니다.
6. 분석 결과를 목표와 방법을 포함한 `user_request`로 정리합니다.
7. 분석 결과를 `messages`에 추가하고 현재 상태를 저장합니다.
8. 갱신된 `messages`와 `user_request`를 반환하고, `ai_recommendation`을 초기화합니다.

**목차 검토 에이전트와의 연계**

| 에이전트 | 역할 |
|---|---|
| `outline_reviewer` | 목차의 문제점과 개선 방향을 검토하여 `ai_recommendation`에 저장합니다. |
| `business_analyst` | 검토 의견과 사용자 발언을 함께 분석하여 현재 필요한 작업을 정리합니다. |
| `supervisor` | 분석 결과를 참고하여 다음 작업과 담당 에이전트를 선택합니다. |

**참고**

- `ai_recommendation`이 없는 경우에도 사용자 발언과 현재 작업 상황을 바탕으로 요구사항을 분석합니다.
- 현재 구현은 이전 `user_request`를 입력 데이터에 준비하지만, 시스템 프롬프트에는 해당 변수를 직접 포함하지 않습니다.
- `outline_reviewer`의 검토 결과가 그래프 실행 중 전달되려면 리뷰어 노드와 연결을 그래프에 추가해야 합니다.

In [5]:
def business_analyst(state: State): #
    print("\n\n============ BUSINESS ANALYST ============")

    # (1) 시스템 프롬프트 정의
    business_analyst_system_prompt = PromptTemplate.from_template(
        """
        너는 책을 쓰는 AI팀의 비즈니스 애널리스트로서, 
        AI팀의 진행상황과 "사용자 요구사항"을 토대로,
        현 시점에서 'ai_recommendation'과 최근 사용자의 발언을 바탕으로 요구사항이 무엇인지 판단한다.
        지난 요청사항이 달성되었는지 판단하고, 현 시점에서 어떤 작업을 해야 하는지 결정한다.

        다음과 같은 템플릿 형태로 반환한다. 
        ```
        - 목표: OOOO \n 방법: OOOO
        ```

        ------------------------------------
        *AI 추천(ai_recommendation)* : {ai_recommendation}
        ------------------------------------
        사용자 최근 발언: {user_last_comment}
        ------------------------------------
        참고자료: {references}
        ------------------------------------
        목차 (outline): {outline}
        ------------------------------------
        "messages": {messages}
        """
    )
    
    # (2) 시스템 프롬프트와 모델을 연결
    ba_chain = business_analyst_system_prompt | llm | StrOutputParser()

    #(3) 상태에서 메시지를 가져옴
    messages = state["messages"]

    # (3) 사용자의 마지막 발언을 가져옴
    user_last_comment = None
    for m in messages[::-1]:
        if isinstance(m, HumanMessage):
            user_last_comment = m.content
            break

    # (3) 입력 자료 준비
    inputs = {
        "ai_recommendation": state.get("ai_recommendation", None),
        "previous_user_request": state.get("user_request", None),
        "references": state.get("references", {"queries": [], "docs": []}),
        "outline": get_outline(current_path),
        "messages": messages,
        "user_last_comment": user_last_comment
    }

    # (4) 시스템 프롬프트를 통해 사용자 요구사항을 분석
    user_request = ba_chain.invoke(inputs)

    #⑥ (5) businessage analyst의 결과를 메시지에 추가
    business_analyst_message = f"[Business Analyst] {user_request}"
    print(business_analyst_message)
    messages.append(AIMessage(business_analyst_message))

    save_state(current_path, state) # (6) 현재 state 내용 저장

    return {
        "messages": messages,
        "user_request": user_request,
        "ai_recommendation": ""
    }

# chap 15 - 신규 에이전트 추가

## outline_reviewer() — 목차 검토 및 개선 방향 제안

콘텐츠 전략가가 작성한 목차를 검토하고, 사용자의 요구사항과 참고 자료를 바탕으로 문제점과 개선 방향을 제안하는 에이전트입니다.  
검토 결과는 공유 상태의 `ai_recommendation`에 저장합니다. 비즈니스 분석가는 이 내용을 참고하여 현재 필요한 작업을 판단하고, 분석 결과를 Supervisor에게 전달합니다.


**비즈니스 분석가 에이전트와의 연계**

1. `outline_reviewer`가 작성된 목차를 검토하고 개선 방향을 `ai_recommendation`에 저장합니다.
2. `business_analyst`가 검토 결과와 사용자 발언, 참고 자료, 현재 목차를 함께 살펴봅니다.
3. 비즈니스 분석가는 현재 필요한 작업을 판단하여 `user_request`에 정리합니다.
4. Supervisor가 분석 결과를 참고하여 다음 담당 에이전트를 선택합니다.
5. 비즈니스 분석가는 처리한 `ai_recommendation`을 빈 문자열로 초기화합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 사용자 요구사항, 참고 자료, 대화 내용 및 검토 결과를 공유하는 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `user_request` | 비즈니스 분석가가 정리한 사용자의 요구사항 |
| `references` | 목차 작성과 검토에 활용할 참고 자료 |
| `messages` | 사용자와 에이전트의 대화 및 작업 기록 |
| `ai_recommendation` | 목차 검토 결과와 후속 작업에 대한 제안 |

**주요 검토 항목**

- 목차가 사용자 요구사항을 충족하는지 확인합니다.
- 목차의 논리적인 흐름이 적절한지 검토합니다.
- 근거가 부족한 내용이 있는지 확인합니다.
- 제공된 참고 자료를 충분히 활용했는지 살펴봅니다.
- 참고 자료가 부족하거나 잘못된 부분이 있는지 확인합니다.
- 더미 URL이나 실제 참고 페이지가 아닌 대표 URL을 확인하고, 삭제가 필요한 주소를 제안합니다.
- 그 밖에 개선이 필요한 사항과 후속 작업을 제안합니다.

**반환값**

- `dict`: 검토 메시지가 추가된 `messages`와 검토 결과가 담긴 `ai_recommendation`을 반환합니다.

**주요 처리 과정**

1. 목차 검토 기준과 후속 작업 제안 방식을 시스템 프롬프트에 정의합니다.
2. 공유 상태에서 `user_request`, `references`, `messages`를 가져옵니다.
3. `get_outline()`을 호출하여 저장된 목차를 불러옵니다.
4. 시스템 프롬프트와 LLM을 연결하여 목차 검토 체인을 구성합니다.
5. 사용자 요구사항, 목차, 참고 자료와 대화 내용을 바탕으로 검토 결과를 생성합니다.
6. 검토 결과를 출력하고 `messages`에 추가합니다.
7. 검토 결과를 `ai_recommendation`에 저장하여 반환합니다.

In [6]:
def outline_reviewer(state: State): # (0)
    print("\n\n============ OUTLINE REVIEWER ============")

    # (1) 시스템 프롬프트 정의
    outline_reviewer_system_prompt = PromptTemplate.from_template(
        """
        너는 AI팀의 목차 리뷰어로서, AI팀이 작성한 목차(outline)를 검토하고 문제점을 지적한다. 

        - outline이 사용자의 요구사항을 충족시키는지 여부
        - outline의 논리적인 흐름이 적절한지 여부
        - 근거에 기반하지 않은 내용이 있는지 여부
        - 주어진 참고자료(references)를 충분히 활용했는지 여부
        - 참고자료가 충분한지, 혹은 잘못된 참고자료가 있는지 여부
        - example.com 같은 더미 URL이 있는지 여부: 
        - 실제 페이지 URL이 아닌 대표 URL로 되어 있는 경우 삭제 해야함: 어떤 URL이 삭제되어야 하는지 명시하라.
        - 기타 리뷰 사항

        그 분석 결과를 설명하고, 다음 어떤 작업을 하면 좋을지 제안하라.
        
        - 분석결과: outline이 사용자의 요구사항을 충족시키는지 여부
        - 제안사항: (vector_search_agent, communicator 중 어떤 agent를 호출할지)

        ------------------------------------------
        user_request: {user_request}
        ------------------------------------------
        references: {references}
        ------------------------------------------
        outline: {outline}
        ------------------------------------------
        messages: {messages}
        """
    )

    # (2) inputs에 들어갈 내용 정리    
    user_request = state.get("user_request", None)
    outline = get_outline(current_path)
    references = state.get("references", {"queries": [], "docs": []})
    messages = state.get("messages", [])

    inputs = {
        "user_request": user_request,
        "outline": outline,
        "references": references,
        "messages": messages
    }

    # 시스템 프롬프트와 모델을 연결
    outline_reviewer_chain = outline_reviewer_system_prompt | llm

    # (3) 목차 리뷰
    review = outline_reviewer_chain.stream(inputs)

    gathered = None

    for chunk in review:
        print(chunk.content, end='')

        if gathered is None:
            gathered = chunk
        else:
            gathered += chunk
            
    # (4) outline_review 에이전트의 작업 후기를 메시지에 추가
    if '[OUTLINE REVIEW AGENT]' not in gathered.content:
        gathered.content = f"[OUTLINE REVIEW AGENT] {gathered.content}"

    print(gathered.content)
    messages.append(gathered)

    # (5) ai_recommendation은 목차 리뷰 결과를 사용
    ai_recommendation = gathered.content

    return {"messages": messages, "ai_recommendation": ai_recommendation}

# Chap 15 - 기능 수정: 무한 루프 방지하기

## supervisor() — AI 에이전트 작업 관리 및 반복 호출 제한

AI 에이전트 팀의 작업을 관리하고 현재 수행할 작업과 담당 에이전트를 결정하는 함수입니다. 기존 대화 내용과 작성된 목차를 참고하여 다음 작업을 `Task` 객체로 생성하고 작업 이력에 추가합니다.  
에이전트들이 계속 작업을 주고받는 상황을 줄이기 위해 `supervisor_call_count`로 Supervisor의 호출 횟수를 추적합니다. 호출 횟수가 기준을 초과하면 다음 작업을 `communicator`로 지정하여 사용자에게 진행 상황을 보고하도록 합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 에이전트의 현재 실행 상태를 담은 딕셔너리 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자 및 AI 에이전트 간 대화 메시지 |
| `task_history` | 이전에 생성된 작업 이력 |
| `supervisor_call_count` | Supervisor의 반복 호출을 추적하는 정수 |

**반환값**

- `dict`: 갱신된 `messages`, `task_history`, `supervisor_call_count`를 반환합니다.
- 반환된 데이터는 LangGraph의 공유 상태에 반영됩니다.

**주요 처리 과정**

1. `PromptTemplate`을 사용하여 Supervisor의 시스템 프롬프트를 정의합니다.
2. `llm.with_structured_output(Task)`를 사용하여 LLM의 출력을 `Task` 객체로 구조화합니다.
3. 기존 대화 메시지와 `get_outline()`으로 읽어 온 목차를 입력 데이터로 준비합니다.
4. 상태에서 `supervisor_call_count`를 가져옵니다. 값이 없으면 `0`으로 사용합니다.
5. 호출 횟수가 `2` 이하이면 LLM을 호출하여 다음 작업과 담당 에이전트를 결정합니다.
6. 호출 횟수가 `2`를 초과하면 진행 상황을 사용자에게 보고하도록 `communicator`의 `Task`를 생성합니다.
7. 생성된 `Task`를 `task_history`에 추가하고, 작업 결정 내용을 `AIMessage`로 만들어 `messages`에 추가합니다.
8. 호출 횟수에 `1`을 더해 갱신된 상태와 함께 반환합니다.

**반복 호출 제한 예시**

초기 호출 횟수가 `0`이고 중간에 초기화되지 않는 경우입니다.

| Supervisor 실행 | 실행 전 호출 횟수 | 동작 |
|---|---:|---|
| 첫 번째 | 0 | LLM이 다음 작업을 선택합니다. |
| 두 번째 | 1 | LLM이 다음 작업을 선택합니다. |
| 세 번째 | 2 | LLM이 다음 작업을 선택합니다. |
| 네 번째 | 3 | `communicator` 작업을 생성합니다. |

**참고**

- Supervisor는 직접 웹 검색이나 목차 작성을 수행하지 않고 다음 작업을 결정합니다.
- `Task.agent`는 `content_strategist`, `communicator`, `web_search_agent`, `vector_search_agent` 중 하나입니다.
- Supervisor는 호출 횟수가 기준 이하인 경우에도 필요하다고 판단하면 `communicator`를 선택할 수 있습니다.
- `communicator` 실행 후 호출 횟수를 `0`으로 초기화하면, 다음 사용자 입력 이후에도 에이전트들이 다시 작업을 이어갈 수 있습니다.

In [7]:
from langchain_core.messages import AIMessage
from langchain_core.prompts import PromptTemplate
#from utils import get_outline 
# #%run ./utils.ipynb로 불러온 get_outline 사용

def supervisor(state: State): # supervisor 에이전트 추가
    print("\n\n============ SUPERVISOR ============")

    # 시스템 프롬프트 정의
    supervisor_system_prompt = PromptTemplate.from_template(
        """
        너는 AI 팀의 supervisor로서 AI 팀의 작업을 관리하고 지도한다.
        사용자가 원하는 책을 써야 한다는 최종 목표를 염두에 두고, 
        사용자의 요구를 달성하기 위해 현재 해야할 일이 무엇인지 결정한다.

        supervisor가 활용할 수 있는 agent는 다음과 같다.     
        - content_strategist: 사용자의 요구사항이 명확해졌을 때 사용한다. AI 팀의 콘텐츠 전략을 결정하고, 전체 책의 목차(outline)를 작성한다. 
        - communicator: AI 팀에서 해야 할 일을 스스로 판단할 수 없을 때 사용한다. 사용자에게 진행상황을 사용자에게 보고하고, 다음 지시를 물어본다. 
        - web_search_agent: vector_search_agent를 시도하고, 검색 결과(references)에 필요한 정보가 부족한 경우 사용한다. 웹 검색을 통해 해당 정보를 Vector DB에 보강한다. 
        - vector_search_agent: 목차 작성을 위해 필요한 자료를 확보하기 위해 벡터 DB 검색을 한다. 

        아래 내용을 고려하여, 현재 해야할 일이 무엇인지, 사용할 수 있는 agent를 단답으로 말하라.

        ------------------------------------------
        previous_outline: {outline}
        ------------------------------------------
        messages:
        {messages}
        """
    )

    # 체인 연결
    supervisor_chain = supervisor_system_prompt | llm. with_structured_output(Task)    

    # 메시지 가져오기
    messages = state.get("messages", [])

    # inputs 설정
    inputs = {
        "messages": messages,
        "outline": get_outline(current_path)
    }

    supervisor_call_count = state.get("supervisor_call_count", 0)

    if supervisor_call_count > 2:
        print("Supervisor 호출 횟수 초과: Communicator 호출")
        task = Task(
            agent="communicator", 
            done=False, 
            description="supervisor 호출 횟수 초과했으므로, 현재까지의 진행상황을 사용자에게 보고한다. ",
            done_at="", 
        )
    else:
        task = supervisor_chain.invoke(inputs)

    task_history = state.get("task_history", [])    # 작업 이력 가져오기
    task_history.append(task)                    	# 작업 이력에 추가
   
    # 메시지 추가
    supervisor_message = AIMessage(f"[Supervisor] {task}")
    messages.append(supervisor_message)
    print(supervisor_message.content)

    # state 업데이트
    return {
        "messages": messages, 
        "task_history": task_history, 
        "supervisor_call_count": supervisor_call_count + 1
    }

## supervisor_router() — 에이전트 실행 경로 결정
Supervisor가 생성한 가장 최근 작업을 확인하고, 해당 작업을 수행할 에이전트의 이름을 반환하는 함수입니다.  
LangGraph의 조건부 분기(Conditional Edge)에서 다음 실행 노드를 선택하는 데 사용됩니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 작업 이력이 포함된 현재 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `task_history` | Supervisor가 생성한 작업 이력 |
| `task_history[-1]` | 가장 최근에 생성된 작업 |
| `task_history[-1].agent` | 해당 작업을 담당할 에이전트 |

**반환값**

- `str`: 다음에 실행할 에이전트 이름을 반환합니다.
- 반환 가능한 값은 `Task.agent`에 정의된 네 가지 에이전트 이름입니다.

**주요 처리 과정**

1. `state["task_history"]`에서 작업 이력을 가져옵니다.
2. `[-1]` 인덱스를 사용하여 가장 최근에 생성된 `Task` 객체를 선택합니다.
3. 해당 작업의 `agent` 값을 반환합니다.
4. LangGraph는 반환된 이름을 기준으로 다음 실행 노드를 선택합니다.

In [8]:
def supervisor_router(state: State):
    task = state['task_history'][-1]
    return task.agent			


# Chap 15 - 기능 수정: 벡터 검색 에이전트도 비즈니스분석가 에이전트에게 조언하도록구성하기

## vector_search_agent() — 벡터 검색 및 후속 작업 제안

벡터 데이터베이스에서 목차 작성과 개선에 필요한 문서를 검색하는 함수입니다. 검색 쿼리와 문서를 공유 상태의 `references`에 저장하고, 중복 문서를 제거합니다.  
검색이 끝나면 비즈니스 분석가가 참고할 수 있도록 `ai_recommendation`을 작성합니다. 추천 내용은 참고 자료가 목차 개선에 충분한지 확인하고, 충분하다면 `content_strategist`가 목차를 작성하도록 제안합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 작업 이력, 대화 내용, 참고 자료 및 AI의 추천 내용을 담은 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자와 에이전트의 대화 및 작업 기록 |
| `task_history` | 작업 이력과 현재 수행할 검색 작업 |
| `references` | 검색 쿼리와 수집한 참고 문서 |
| `references["queries"]` | 벡터 검색에 사용한 쿼리 목록 |
| `references["docs"]` | 검색된 문서 객체 목록 |
| `ai_recommendation` | 검색 이후 참고 자료 확인과 후속 작업에 관한 제안 |

**반환값**

- `dict`: 갱신된 `messages`, `task_history`, `references`, `ai_recommendation`을 반환합니다.
- 검색 결과와 후속 작업 제안은 공유 상태를 통해 `business_analyst`에 전달됩니다.

**주요 처리 과정**

1. `task_history`에서 최근 작업을 가져오고, 담당 에이전트가 `vector_search_agent`인지 확인합니다.
2. 작업 설명, 기존 검색 기록, 대화 내용과 현재 목차를 검색 프롬프트에 반영합니다.
3. `llm.bind_tools([retrieve])`를 사용하여 LLM에 벡터 검색 도구를 연결합니다.
4. LLM을 호출하여 검색에 필요한 Tool Call을 생성합니다.
5. 각 Tool Call에 대해 `retrieve()`를 실행하고 검색 쿼리와 결과 문서를 `references`에 추가합니다.
6. 문서의 `page_content`를 기준으로 중복 문서를 제거합니다.
7. 검색 결과를 출력하고 현재 작업의 완료 상태와 완료 시각을 기록합니다.
8. 검색 완료 메시지를 `messages`에 추가합니다.
9. 참고 자료의 충분성을 확인하고, 자료가 충분하면 `content_strategist`의 목차 작성을 제안하는 내용을 `ai_recommendation`에 저장합니다.
10. 갱신된 상태를 반환하고, 그래프 연결에 따라 `business_analyst`로 이동합니다.

**후속 작업 판단**

- `business_analyst`는 검색 결과와 `ai_recommendation`을 함께 참고하여 현재 필요한 작업을 분석합니다.
- 참고 자료가 충분하다면 목차 작성이나 개선을 진행할 수 있습니다.
- 자료가 부족하다면 분석 결과를 바탕으로 Supervisor가 다음 작업과 담당 에이전트를 선택합니다.
- 검색 종료 후 `communicator`의 작업을 직접 생성하지 않습니다.

**참고**

- `ai_recommendation`은 자료가 충분한지 **확인하라는 제안**입니다. `vector_search_agent`가 자료의 충분성을 직접 판정하는 것은 아닙니다.
- 다음 담당 에이전트의 최종 선택은 Supervisor가 수행합니다.


In [9]:
# from tools import retrieve 
# #%run ./tools.ipynb로 불러온 retrieve 사용
from datetime import datetime

def vector_search_agent(state: State):
    print("\n\n============ VECTOR SEARCH AGENT ============")
    
    tasks = state.get("task_history", [])
    task = tasks[-1]
    if task.agent != "vector_search_agent":
        raise ValueError(f"Vector Search Agent가 아닌 agent가 Vector Search Agent를 시도하고 있습니다.\n {task}")

    vector_search_system_prompt = PromptTemplate.from_template(
        """
        너는 다른 AI Agent 들이 수행한 작업을 바탕으로, 
        목차(outline) 작성에 필요한 정보를 벡터 검색을 통해 찾아내는 Agent이다.

        현재 목차(outline)을 작성하는데 필요한 정보를 확보하기 위해, 
        다음 내용을 활용해 적절한 벡터 검색을 수행하라. 

        - 검색 목적: {mission}
        --------------------------------
        - 과거 검색 내용: {references}
        --------------------------------
        - 이전 대화 내용: {messages}
        --------------------------------
        - 목차(outline): {outline}
        """
    )

    # inputs 설정
    mission = task.description
    references = state.get("references", {"queries": [], "docs": []})
    messages = state["messages"]
    outline = get_outline(current_path)

    inputs = {
        "mission": mission,
        "references": references,
        "messages": messages,
        "outline": outline
    }

    # LLM과 벡터 검색 모델 연결
    llm_with_retriever = llm.bind_tools([retrieve]) 
    vector_search_chain = vector_search_system_prompt | llm_with_retriever

    # LLM과 벡터 검색 모델 연결
    search_plans = vector_search_chain.invoke(inputs)

    # 검색할 내용 출력
    for tool_call in search_plans.tool_calls:
        print('-----------------------------------', tool_call)
        args = tool_call["args"]
       
        query = args["query"] 
        retrieved_docs = retrieve.invoke(args)
		# (1) 결과 담아 두기
        references["queries"].append(query) 
        references["docs"] += retrieved_docs
    
    unique_docs = []
    unique_page_contents = set()

    for doc in references["docs"]:
        if doc.page_content not in unique_page_contents:
            unique_docs.append(doc)
            unique_page_contents.add(doc.page_content)

    references["docs"] = unique_docs

    # 검색 결과 출력 – 쿼리 출력
    print('Queries:--------------------------')
    queries = references["queries"]
    for query in queries:
        print(query)
    
    # 검색 결과 출력 – 문서 청크 출력
    print('References:--------------------------')
    for doc in references["docs"]:
        print(doc.page_content[:100])
        print('--------------------------')

    # task 완료
    tasks[-1].done = True
    tasks[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # # communicator로 보내는 Task 생성 코드 삭제
    # new_task = Task(
    #     agent="communicator",
    #     done=False,
    #     description="AI팀의 진행상황을 사용자에게 보고하고, 사용자의 의견을 파악하기 위한 대화를 나눈다",
    #     done_at=""
    # )
    # tasks.append(new_task)

    # vector search agent의 작업후기를 메시지로 생성
    msg_str = f"[VECTOR SEARCH AGENT] 다음 질문에 대한 검색 완료: {queries}"
    message = AIMessage(msg_str)
    print(msg_str)

    messages.append(message)
    ai_recommendation = "현재 참고자료(references)가 목차(outline)를 개선하는데 충분한지 확인하라. 충분하다면 content_strategist로 목차 작성을 하라. "
    
    # state 업데이트
    return {
        "messages": messages,
        "task_history": tasks,
        "references": references,
        "ai_recommendation": ai_recommendation
    }

## web_search_agent() — 웹 검색 및 Chroma DB 데이터 수집

웹 검색을 통해 책의 목차 작성에 필요한 외부 자료를 수집하는 함수입니다.  
LLM이 생성한 검색 쿼리를 바탕으로 웹 검색을 수행하고, 수집된 데이터를 Chroma 벡터 데이터베이스에 추가합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 현재 작업, 대화 메시지, 기존 참고 자료를 포함하는 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 기존 대화 및 에이전트 실행 메시지 |
| `task_history` | 현재 웹 검색 작업 및 이전 작업 이력 |
| `references` | 기존 검색 기록과 참고 자료 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 웹 검색 결과는 별도 함수를 통해 Chroma DB에 추가됩니다.

**주요 처리 과정**

1. 가장 최근 작업을 가져와 담당 에이전트가 `web_search_agent`인지 확인합니다.
2. 작업 설명, 기존 검색 기록, 대화 내용, 목차, 현재 시간을 포함한 시스템 프롬프트를 구성합니다.
3. `llm.bind_tools([web_search])`를 통해 LLM에 웹 검색 도구를 연결합니다.
4. `web_search_chain.invoke()`를 호출하여 검색 Tool Call을 생성합니다.
5. 각 Tool Call에 포함된 검색 쿼리를 `queries` 리스트에 저장합니다.
6. `web_search.invoke()`로 웹 검색을 실행하고 검색 결과가 저장된 JSON 파일 경로를 가져옵니다.
7. `add_web_pages_json_to_chroma()`를 호출하여 해당 JSON 데이터를 Chroma DB에 추가합니다.
8. 현재 작업을 완료 처리하고 완료 시간을 기록합니다.
9. 새로 수집한 자료를 벡터 검색할 수 있도록 `vector_search_agent`의 작업을 생성합니다.
10. 웹 검색 완료 메시지를 `messages`에 추가하고 갱신된 상태를 반환합니다.

In [10]:
#from tools import web_search, add_web_pages_json_to_chroma 
# #%run ./tools.ipynb로 불러온 web_search, add_web_pages_json_to_chroma 사용

def web_search_agent(state: State): #① (0)
    print("\n\n============ WEB SEARCH AGENT ============")

    # 작업 리스트 가져와서 web search agent 가 할 일인지 확인하기
    tasks = state.get("task_history", [])
    task = tasks[-1]

    if task.agent != "web_search_agent":
        raise ValueError(f"Web Search Agent가 아닌 agent가 Web Search Agent를 시도하고 있습니다.\n {task}")
    
    #③ 시스템 프롬프트 정의
    web_search_system_prompt = PromptTemplate.from_template(
        """
        너는 다른 AI Agent 들이 수행한 작업을 바탕으로, 
        목차(outline) 작성에 필요한 정보를 웹 검색을 통해 찾아내는 Web Search Agent이다.

        현재 부족한 정보를 검색하고, 복합적인 질문은 나눠서 검색하라.

        - 검색 목적: {mission}
        --------------------------------
        - 과거 검색 내용: {references}
        --------------------------------
        - 이전 대화 내용: {messages}
        --------------------------------
        - 목차(outline): {outline}
        --------------------------------
        - 현재 시각 : {current_time}
        """
    )
    
    #④ 기존 대화 내용 가져오기
    messages = state.get("messages", [])

    #⑤ 인풋 자료 준비하기
    inputs = {
        "mission": task.description,
        "references": state.get("references", {"queries": [], "docs": []}),
        "messages": messages,
        "outline": get_outline(current_path),
        "current_time": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    #⑥ LLM과 웹 검색 모델 연결
    llm_with_web_search = llm.bind_tools([web_search])

    #⑦ 시스템 프롬프트와 모델을 연결
    web_search_chain = web_search_system_prompt | llm_with_web_search

    #⑧ 웹 검색 tool_calls 가져오기
    search_plans = web_search_chain.invoke(inputs)

    #⑨ 어떤 내용을 검색했는지 담아두기
    queries = []

    #⑩ 검색 계획(tool_calls)에 따라 검색하기
    for tool_call in search_plans.tool_calls:
        print('-------- web search --------', tool_call)
        args = tool_call["args"]
        
        queries.append(args["query"])

        # (10)  검색 결과를 chroma에 추가
        _, json_path = web_search.invoke(args)
        print('json_path:', json_path)

        # (10)  JSON 파일을 chroma에 추가
        add_web_pages_json_to_chroma(json_path)

    #⑪ (11) task 완료
    tasks[-1].done = True
    tasks[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    #⑪ (11) 새로운 task 추가
    task_desc = "AI팀이 쓸 책의 세부 목차를 결정하기 위한 정보를 벡터 검색을 통해 찾아낸다."
    task_desc += f" 다음 항목이 새로 추가되었다\n: {queries}"
    
    new_task = Task(
        agent="vector_search_agent",
        done=False,
        description=task_desc,
        done_at=""
    )

    tasks.append(new_task)

    #⑫ (12) 작업 후기 메시지
    msg_str = f"[WEB SEARCH AGENT] 다음 질문에 대한 검색 완료: {queries}"
    messages.append(AIMessage(msg_str))

    #⑬ (13) state 업데이트
    return {
        "messages": messages,
        "task_history": tasks
    }

# chap 15 - 목차 작성 템플릿을 활용해 시스템 프롬프트 발전시키기

## content_strategist() — 목차 작성 템플릿을 활용한 책의 목차 작성 및 수정

목차 작성 템플릿의 구체적인 규칙과 사용자의 요구사항을 반영하여 책의 목차를 작성하거나 수정하는 함수입니다.

비즈니스 분석가가 사용자 의도를 분석해 정리한 `user_request`를 시스템 프롬프트에 반영합니다. 이를 통해 작성할 목차의 방향이 사용자의 요청과 일치하도록 합니다. 생성된 목차는 Markdown 파일로 저장합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 대화 내용, 작업 이력, 참고 자료 및 사용자 요구사항이 포함된 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자와 AI 에이전트의 대화 내용 |
| `task_history` | 수행 중인 작업 및 이전 작업 이력 |
| `references` | 목차 작성에 활용할 참고 자료 |
| `references["queries"]` | 참고 자료 검색에 사용한 쿼리 |
| `references["docs"]` | 검색으로 확보한 참고 문서 |
| `user_request` | 비즈니스 분석가가 사용자 의도를 분석하여 정리한 요청 내용 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 저장 파일: `{current_path}/data/outline.md`

**주요 처리 과정**

1. 목차 작성 템플릿을 바탕으로 목차의 구성과 작성에 필요한 구체적인 규칙을 시스템 프롬프트에 추가합니다.
2. 비즈니스 분석가가 정리한 `user_request`를 프롬프트에 반영하고, 목차 작성 시 이를 준수하도록 지시합니다.
3. `PromptTemplate`, LLM, `StrOutputParser`를 연결하여 목차 생성 체인을 구성합니다.
4. `get_outline()`을 호출하여 기존에 작성된 목차를 불러옵니다.
5. 기존 목차, 사용자 대화, 참고 자료와 `user_request`를 입력 데이터로 준비합니다.
6. `content_strategist_chain.stream()`을 사용하여 목차를 스트리밍 방식으로 생성합니다.
7. 생성된 텍스트를 `gathered` 문자열에 누적합니다.
8. `save_outline()`을 호출하여 목차를 Markdown 파일로 저장합니다.
9. 목차 작성 완료 메시지를 `messages`에 추가합니다.
10. 현재 작업이 `content_strategist`의 작업인지 확인하고 완료 상태와 시간을 기록합니다.
11. 다음 작업인 `communicator`의 `Task`를 생성하고 상태를 반환합니다.

**참고**

- 목차 작성 템플릿은 에이전트가 따라야 할 목차 구성과 작성 규칙을 구체화하는 데 사용합니다.
- `user_request`에는 원문을 그대로 옮긴 내용뿐 아니라 비즈니스 분석가가 파악한 사용자 의도가 담깁니다.
- 기존 목차가 있는 경우 이를 참고하여 사용자의 요구사항에 맞게 수정합니다.

In [11]:
from langchain_core.output_parsers.string import StrOutputParser
# from utils import save_outline
# #%run ./utils.ipynb로 불러온 save_outline 사용

def content_strategist(state: State):
    print("\n\n============ CONTENT STRATEGIST ============")
    
    task_history = state.get("task_history", []) # 작업 이력 가져오기 (아래에 있던 코드 위로 이동)
    task = task_history[-1]
    if task.agent != "content_strategist":
        raise ValueError(f"Content Strategist가 아닌 agent가 목차 작성을 시도하고 있습니다.\n {task}")
    
    # 시스템 프롬프트 정의
    content_strategist_system_prompt = PromptTemplate.from_template(
        """
        너는 책을 쓰는 AI팀의 콘텐츠 전략가(Content Strategist)로서,
        이전 대화 내용을 바탕으로 사용자의 요구사항을 분석하고, AI팀이 쓸 책의 세부 목차를 결정한다.

        지난 목차가 있다면 그 버전을 사용자의 요구에 맞게 수정하고, 없다면 새로운 목차를 제안한다.
        목차를 작성하는데 필요한 정보는 "참고 자료"에 있으므로 활용한다. 
        
        다음 정보를 활용하여 목차를 작성하라. 
        - 사용자 요구사항(user_request)
        - 작업(task)
        - 검색 자료 (references)
        - 기존 목차 (previous_outline)
        - 이전 대화 내용(messages)

        너의 작업 목표는 다음과 같다:
        1. 만약 "기존 목차 구조 (previous_outline)"이 존재한다면, 사용자의 요구사항을 토대로 "기존 목차 구조"에서 어떤 부분을 수정하거나 추가할지 결정한다.
        - "이번 목차 작성의 주안점"에 사용자 요구사항(user_request)을 충족시키는 것을 명시해야 한다.
        2. 책의 전반적인 구조(chapter, section)를 설계하고, 각 chpater와 section의 제목을 정한다.
        3. 책의 전반적인 세부구조(chapter, section, sub-section)를 설계하고, sub-section 하부의 주요 내용을 리스트 형태로 정리한다.
        4. 목차의 논리적인 흐름이 사용자 요구를 충족시키는지 확인한다.
        5. 참고자료 (references)를 적극 활용하여 근거에 기반한 목차를 작성한다.
        6. 참고문헌은 반드시 참고자료(references) 자료를 근거로 작성해야 하며, 최대한 풍부하게 준비한다. URL은 전체 주소를 적어야 한다.
        7. 추가 자료나 리서치가 필요한 부분을 파악하여 supervisor에게 요청한다.

        사용자 요구사항(user_request)을 최우선으로 반영하는 목차로 만들어야 한다. 

        --------------------------------
        - 사용자 요구사항(user_request): 
        {user_request}
        --------------------------------
        - 작업(task): 
        {task}
        --------------------------------
        - 참고 자료 (references)
        {references}
        --------------------------------
        - 기존 목차 (previous_outline)
        {outline}
        --------------------------------
        - 이전 대화 내용(messages)
        {messages}
        --------------------------------

        작성 형식 아래 양식을 지키되 하부 항목으로 더 세분화해도 좋다. 목차(outline) 양식의 챕터, 섹션 등 항목의 갯수는 필요한만큼 추가하라. 
        섹션 갯수는 최소 2개 이상이어야 하며, 더 많으면 좋다. 

        outline_template은 예시로 앞부분만 제시한 것이다. 각 장은 ':---CHAPTER DIVIDER---:'로 구분한다.
        outline_template:
        {outline_template}

        사용자가 추가 피드백을 제공할 수 있도록 논리적인 흐름과 주요 목차 아이디어를 제안하라.    
        """
    )

    # 시스템 프롬프트와 모델을 연결
    content_strategist_chain = content_strategist_system_prompt | llm | StrOutputParser()

    user_request = state.get("user_request", "") # 사용자 요구사항 가져오기

    messages = state["messages"]        # 상태에서 메시지를 가져옴
    outline = get_outline(current_path) # 저장된 목차를 가져옴

    with open(f"{current_path}/templates/outline_template.md", "r", encoding='utf-8') as f:
        outline_template = f.read()  

    # 입력값 정의
    inputs = {
        "user_request": user_request,  #사용자 요구사항 user_request
        "task": task,
        "messages": messages,
        "outline": outline, 
        "references": state.get("references", {"queries": [], "docs": []}),
        "outline_template": outline_template # 템플릿 이용하기
    }

    # 목차 작성
    gathered = ''
    for chunk in content_strategist_chain.stream(inputs):
        gathered += chunk
        print(chunk, end='')

    print()

    save_outline(current_path, gathered) # 목차 저장

    # 템플릿을 활용한 작업 후기 메시지 찾기
    if '-----: DONE :-----' in gathered:
        review = gathered.split('-----: DONE :-----')[1]
    else: 
        review = gathered[-200:]


    # 메시지 추가    
    content_strategist_message = f"[Content Strategist] 목차 작성 완료: outline 작성 완료\n {review}"
    print(content_strategist_message)
    messages.append(AIMessage(content_strategist_message))

    # task_history = state.get("task_history", []) # task_history 가져오기
    # # 최근 task 작업완료(done) 처리하기
    # if task_history[-1].agent != "content_strategist": 
    #     raise ValueError(f"Content Strategist가 아닌 agent가 목차 작성을 시도하고 있습니다.\n {task_history[-1]}")
    
    task_history[-1].done = True
    task_history[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # # 다음 작업이 communicator로 사용자와 대화하는 것이므로 새 작업 추가 
    # new_task = Task(
    #     agent="communicator",
    #     done=False,
    #     description="AI팀의 진행상황을 사용자에게 보고하고, 사용자의 의견을 파악하기 위한 대화를 나눈다",
    #     done_at=""
    # )
    # task_history.append(new_task)
    # print(new_task)

    # 현재 state를 업데이트한다. 
    return {
        "messages": messages,
        "task_history": task_history
    }

## communicator() — 사용자 커뮤니케이션 및 진행 상황 보고

AI 에이전트 팀의 작업 진행 상황을 사용자에게 보고하고, 사용자와 대화를 수행하는 함수입니다.  
기존 대화와 작성된 목차를 참고하여 응답을 생성하고, 이를 공유 상태에 기록합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 대화 메시지 및 작업 이력이 포함된 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자와 에이전트 간의 대화 기록 |
| `task_history` | 현재 커뮤니케이션 작업 및 이전 작업 이력 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 생성된 사용자 응답은 콘솔에도 출력됩니다.

**주요 처리 과정**

1. 사용자와 대화하는 커뮤니케이터 역할의 시스템 프롬프트를 정의합니다.
2. `PromptTemplate`과 LLM을 연결하여 응답 생성 체인을 구성합니다.
3. 기존 대화 메시지와 `get_outline()`으로 읽은 목차를 입력 데이터로 준비합니다.
4. `system_chain.stream()`을 통해 LLM 응답을 스트리밍 방식으로 생성합니다.
5. 생성되는 응답을 콘솔에 출력하면서 `gathered`에 누적합니다.
6. 완성된 응답 메시지를 `messages`에 추가합니다.
7. 현재 작업이 `communicator`의 작업인지 확인합니다.
8. 해당 작업을 완료 처리하고 완료 시간을 기록합니다.
9. 갱신된 상태 데이터를 반환합니다.

**참고**
- 기존 목차는 LLM이 응답을 생성할 때 참고 자료로 사용합니다.
- 시스템 프롬프트에서 목차를 다시 출력하지 않도록 지시합니다.

In [12]:
def communicator(state: State):
    print("\n\n============ COMMUNICATOR ============")

    # 시스템 프롬프트 정의
    communicator_system_prompt = PromptTemplate.from_template(
        """
        너는 책을 쓰는 AI팀의 커뮤니케이터로서, 
        AI팀의 진행상황을 사용자에게 보고하고, 사용자의 의견을 파악하기 위한 대화를 나눈다. 

        사용자도 outline(목차)을 이미 보고 있으므로, 다시 출력할 필요는 없다.
        outline: {outline} 
        --------------------------------
        messages: {messages}
        """
    )

    # 시스템 프롬프트와 모델을 연결
    system_chain = communicator_system_prompt | llm

    # 상태에서 메시지를 가져옴
    messages = state["messages"]

    # 입력값 정의
    inputs = {
        "messages": messages,
        "outline": get_outline(current_path)
    }

    # 스트림되는 메시지를 출력하면서, gathered에 모으기
    gathered = None

    print('\nAI\t: ', end='')
    for chunk in system_chain.stream(inputs):
        print(chunk.content, end='')

        if gathered is None:
            gathered = chunk
        else:
            gathered += chunk

    messages.append(gathered)

    task_history = state.get("task_history", []) 
    if task_history[-1].agent != "communicator":
        raise ValueError(f"Communicator가 아닌 agent가 대화를 시도하고 있습니다.\n {task_history[-1]}")
    
    task_history[-1].done = True
    task_history[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    return {
        "messages": messages,
        "task_history": task_history,
        "supervisor_call_count": 0
    }

# Chap 15 - 스스로 판단하고 작업을 이어가는 멀티 에이전트 시스템 만들기

기존에는 에이전트가 작업을 마칠 때마다 `communicator`가 사용자에게 진행 상황을 보고하고 다음 작업을 물어보았습니다.  
변경된 구조에서는 `outline_reviewer`가 작성된 목차를 검토하고, `business_analyst`가 검토 의견과 검색 결과를 바탕으로 현재 상태를 다시 분석합니다.  
Supervisor는 이 분석을 바탕으로 다음 작업을 담당할 에이전트를 선택하고, 사용자와의 소통이 필요할 때 `communicator`를 선택합니다.  
이를 통해 사용자에게 매번 다음 작업을 묻지 않고 필요한 작업을 이어갈 수 있습니다.

**에이전트 연결 구조**

| 출발 노드 | 다음 노드 | 설명 |
|---|---|---|
| `START` | `business_analyst` | 사용자 요구사항과 현재 상황을 분석합니다. |
| `business_analyst` | `supervisor` | 분석 결과를 바탕으로 다음 작업을 결정합니다. |
| `supervisor` | 담당 에이전트 | `supervisor_router`가 선택한 에이전트로 이동합니다. |
| `content_strategist` | `outline_reviewer` | 작성된 목차를 검토합니다. |
| `outline_reviewer` | `business_analyst` | 목차 검토 결과와 개선 제안을 전달합니다. |
| `web_search_agent` | `vector_search_agent` | 웹에서 수집한 자료를 벡터 검색에 활용합니다. |
| `vector_search_agent` | `business_analyst` | 검색 결과와 참고 자료 확인을 위한 제안을 전달합니다. |
| `communicator` | `END` | 사용자에게 응답한 후 현재 그래프 실행을 종료합니다. |

**주요 처리 과정**

1. `StateGraph(State)`로 공유 상태를 사용하는 그래프를 생성합니다.
2. 비즈니스 분석가, Supervisor, 검색·목차 작성·목차 검토·커뮤니케이션 에이전트를 노드로 등록합니다.
3. 시작 노드를 `business_analyst`에 연결하여 사용자 요구사항을 먼저 분석합니다.
4. 분석 결과를 받은 Supervisor가 `supervisor_router`를 통해 다음 담당 에이전트를 선택합니다.
5. 목차 작성이 끝나면 `outline_reviewer`가 목차를 검토하고, 검토 결과를 `ai_recommendation`에 저장하여 `business_analyst`에 전달합니다.
6. 웹 검색이 끝나면 `vector_search_agent`로 이동해 수집한 자료를 검색합니다. 벡터 검색이 끝나면 검색 결과와 `ai_recommendation`을 `business_analyst`에 전달합니다.
7. `business_analyst`는 검토 의견 또는 검색 결과를 분석하고, Supervisor가 다음 작업을 선택할 수 있도록 결과를 전달합니다.
8. Supervisor가 `communicator`를 선택하면 사용자에게 응답하고 그래프 실행을 종료합니다.

**참고**

- `outline_reviewer`와 `vector_search_agent`는 후속 작업을 제안하며, `business_analyst`가 제안을 분석한 뒤 Supervisor가 다음 에이전트를 선택합니다.
- 사용자에게 확인이 필요한지 판단하는 것은 에이전트의 분석과 Supervisor의 선택에 달려 있습니다.

In [13]:
from langgraph.graph import StateGraph, START, END

# 상태 그래프 정의
graph_builder = StateGraph(State)

# Nodes
graph_builder.add_node("business_analyst", business_analyst)
graph_builder.add_node("supervisor", supervisor)     
graph_builder.add_node("communicator", communicator)
graph_builder.add_node("content_strategist", content_strategist)
graph_builder.add_node("outline_reviewer", outline_reviewer)
graph_builder.add_node("vector_search_agent", vector_search_agent)
graph_builder.add_node("web_search_agent", web_search_agent)

# Edges
graph_builder.add_edge(START, "business_analyst")
graph_builder.add_edge("business_analyst", "supervisor")
graph_builder.add_conditional_edges(
    "supervisor", 
    supervisor_router,
    {
        "content_strategist": "content_strategist",
        "communicator": "communicator",
        "vector_search_agent": "vector_search_agent", 
        "web_search_agent": "web_search_agent"
    }
)

graph_builder.add_edge("content_strategist", "outline_reviewer")
graph_builder.add_edge("outline_reviewer", "business_analyst")
graph_builder.add_edge("web_search_agent", "vector_search_agent") #③
graph_builder.add_edge("vector_search_agent", "business_analyst")
graph_builder.add_edge("communicator", END)

graph = graph_builder.compile()

In [14]:
# graph.get_graph().draw_mermaid_png(output_file_path=absolute_path.replace('.ipynb', '.png'))

In [15]:
from langchain_core.messages import SystemMessage

# 상태 초기화
state = State(
    messages = [
        SystemMessage(
                f"""
            너희 AI들은 사용자의 요구에 맞는 책을 쓰는 작가팀이다.
            사용자가 사용하는 언어로 대화하라.

            현재시각은 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}이다.

            """
        )
    ],
    task_history=[]
)

In [16]:
from langchain_core.messages import HumanMessage
# from utils import save_state
# %run ./utils.ipynb로 불러온 save_state 사용

while True:
    user_input = input("\nUser\t: ").strip()

    if user_input.lower() in ['exit', 'quit', 'q']:
        print("Goodbye!")
        break
    
    state["messages"].append(HumanMessage(user_input))
    state = graph.invoke(state)

    print('\n------------------------------------ MESSAGE COUNT\t', len(state["messages"]))

    save_state(current_path, state) # 현재 state 내용 저장




============ BUSINESS ANALYST ============
[Business Analyst] - 목표: 애플과 삼성전자를 CEO, 경영 전략, 기업 문화, 기업 특성의 관점에서 비교하는 목차 작성 — **달성됨**
- 방법: 요청한 네 가지 주제를 반영한 목차가 이미 작성되었으므로 목차를 다시 만들 필요는 없습니다. 다음 단계로 삼성 비교 범위(삼성전자 또는 삼성그룹)를 확정하고, 예비 참고자료를 검증·보강한 뒤 집필을 진행합니다.


============ SUPERVISOR ============
[Supervisor] agent='communicator' done=False description='목차 작성은 완료되었습니다. 다음 단계에 앞서 비교 범위를 삼성전자와 삼성그룹 전체 중 무엇으로 할지 사용자에게 확인합니다.' done_at=''


============ COMMUNICATOR ============

AI	: 목차 작성은 완료했습니다. 요청하신 네 가지 주제를 중심으로 비교 흐름을 구성했고, ‘삼성’의 범위는 우선 **삼성전자**로 잡았습니다. 참고자료는 아직 실제 열람·검증 전인 예비 조사 목록입니다.

집필에 앞서 비교 범위를 정하고 싶습니다. **삼성전자**와 **삼성그룹 전체** 중 어느 쪽을 다룰까요? 별도 의견이 없으시면, 애플과 사업 비교 기준을 맞추기 쉬운 삼성전자를 기준으로 진행하겠습니다.

원하시면 함께 알려주세요.
- 중심 관점: CEO·경영자 사례 / 조직·사업 구조
- 사례 범위: 스마트폰 중심 / 여러 사업 부문을 고르게 비교
------------------------------------ MESSAGE COUNT	 5
Goodbye!
